# Validación cruzada (k-fold) — BETO baseline (cabeza congelada)
Explora los learning rates más estables vistos en el grid search,
para confirmar con varios folds que el resultado no depende del split particular.

In [ ]:
# TODO = AGREGAR SIN HASHTAGS
!git clone https://github.com/camistrika/BETO_HUMOR.git
%cd BETO_HUMOR
!pip install -e . -q

In [ ]:
!pip install -q torchao --upgrade
!pip install -q transformers peft datasets scikit-learn pyyaml

In [ ]:
# === Setup en Colab (saltar si ya estás corriendo local) ===
import os

if 'COLAB_GPU' in os.environ or os.path.exists('/content'):
    REPO_URL = 'https://github.com/TU-USUARIO/TU-REPO.git'  # <- completar
    REPO_NAME = REPO_URL.split('/')[-1].replace('.git', '')

    if not os.path.exists(f'/content/{REPO_NAME}'):
        !git clone {REPO_URL}

    %cd /content/{REPO_NAME}
    !pip install -q -r requirements.txt
    !pip install -q -e .

    # Subir el CSV manualmente si no está en el repo (datasets pesados no se versionan)
    if not os.path.exists('data/raw/haha_2019_train.csv'):
        from google.colab import files
        print('Subí haha_2019_train.csv:')
        uploaded = files.upload()
        !mkdir -p data/raw
        !mv haha_2019_train.csv data/raw/

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from transformers import AutoTokenizer

from betohumor.config import DataConfig, BetoConfig
from betohumor.utils import set_seed
from betohumor.dataset import load_and_split
from betohumor.models.baseline import build_beto_baseline
from betohumor.hyperparam_search import run_one

## 1. Datos y configuraciones

In [ ]:
data_config = DataConfig()
beto_config = BetoConfig()
set_seed(data_config.seed)

df_train, df_val, df_test = load_and_split(data_config)

# Unimos train+val para repartir en folds. test queda intacto, aparte.
df_full = pd.concat([df_train, df_val]).reset_index(drop=True)

tokenizer = AutoTokenizer.from_pretrained(beto_config.base_model)
label_col = data_config.label_col

## 2. Configuración de la búsqueda
El baseline congelado no usa LoraConfig: la única variable a explorar
es el learning rate.

In [ ]:
LR_VALUES = [1e-4, 5e-4]
N_FOLDS   = 5

params_grid = [{'learning_rate': lr} for lr in LR_VALUES]

def builder(params):
    return build_beto_baseline(beto_config)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=data_config.seed)
params_grid

## 3. Correr la validación cruzada

In [ ]:
all_fold_results = []

for params in params_grid:
    fold_scores = []
    for fold_i, (train_idx, val_idx) in enumerate(skf.split(df_full, df_full[label_col])):
        run_name = f"lr{params['learning_rate']}_fold{fold_i+1}"
        print(f"\n--- {run_name} ---")

        df_fold_train = df_full.iloc[train_idx].reset_index(drop=True)
        df_fold_val   = df_full.iloc[val_idx].reset_index(drop=True)

        output_dir = f"results/checkpoints/cv_baseline/{run_name}"
        macro_f1, trainer = run_one(
            builder, params, beto_config, data_config,
            df_fold_train, df_fold_val, tokenizer, output_dir, seed=data_config.seed,
        )

        fold_scores.append(macro_f1)
        all_fold_results.append({**params, 'fold': fold_i + 1, 'macro_f1': macro_f1})
        print(f"Fold {fold_i+1} Macro F1: {macro_f1:.4f}")

    print(f"\n=== lr={params['learning_rate']} -> Media: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f} ===")

## 4. Resumen y guardado

In [ ]:
df_folds = pd.DataFrame(all_fold_results)
df_folds.to_csv('results/cv_results_baseline.csv', index=False)

df_summary = (
    df_folds
    .groupby('learning_rate')['macro_f1']
    .agg(mean_macro_f1='mean', std_macro_f1='std')
    .reset_index()
    .sort_values('mean_macro_f1', ascending=False)
)
df_summary.to_csv('results/cv_results_baseline_summary.csv', index=False)
df_summary